# Divisão Temporal: Treino e Out-of-Time (OOT)

Neste notebook, vamos realizar a divisão temporal dos dados de transações em:
- **Dataset de Treino (df_treino)**: Usado para treinar o modelo de detecção de lavagem de dinheiro
- **Dataset Out-of-Time (df_oot)**: Usado para validação temporal e cálculo de métricas

## 1. Importação de Bibliotecas

In [13]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## 2. Caminho do Arquivo

## 3. Leitura dos Dados

In [14]:
file_path = r"C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\raw\trans_enriched.csv"

# 1) Validar se o arquivo existe
if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Arquivo não encontrado: {file_path}\n"
        "Confirme o caminho e se o arquivo foi gerado na pasta data/raw."
    )

print(f"Carregando dados de: {file_path}")

# 2) Leitura robusta: tenta separador padrão e, se necessário, ';'
try:
    df = pd.read_csv(file_path, low_memory=False)
except UnicodeDecodeError:
    # Alguns CSVs podem estar em latin1/cp1252
    df = pd.read_csv(file_path, encoding="latin1", low_memory=False)

# Se vier tudo em uma coluna só, geralmente o separador real é ';'
if len(df.columns) == 1:
    try:
        df = pd.read_csv(file_path, sep=';', low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, sep=';', encoding="latin1", low_memory=False)

# 3) Validar presença da coluna Timestamp
if 'Timestamp' not in df.columns:
    raise KeyError(
        "Coluna 'Timestamp' não encontrada. "
        f"Colunas disponíveis: {list(df.columns)}"
    )

# 4) Converter Timestamp para datetime sem quebrar a execução
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', dayfirst=True)

# Avisar se houve datas inválidas
datas_invalidas = df['Timestamp'].isna().sum()
if datas_invalidas > 0:
    print(f"⚠️ {datas_invalidas:,} registros com Timestamp inválido (NaT) após conversão.")

print(f"\nDados carregados com sucesso!")
print(f"Total de registros: {len(df):,}")
print(f"\nPeríodo dos dados:")
print(f"Data inicial: {df['Timestamp'].min()}")
print(f"Data final: {df['Timestamp'].max()}")
print(f"\nPrimeiras linhas:")
df.head()

Carregando dados de: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\raw\trans_enriched.csv
⚠️ 431 registros com Timestamp inválido (NaT) após conversão.

Dados carregados com sucesso!
Total de registros: 5,078,345

Período dos dados:
Data inicial: 2022-01-09 00:00:00
Data final: 2022-12-09 23:51:00

Primeiras linhas:


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,From Bank Name,Bank ID,Account Number,From Entity ID,From Entity Name,To Bank Name,Bank ID_To,Account Number_To,To Entity ID,To Entity Name
0,2022-01-09 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,...,National Bank of Laramie,10,8000EBD30,800D232D0,Partnership #1,National Bank of Laramie,10,8000EBD30,800D232D0,Partnership #1
1,2022-01-09 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,...,Sappo Cooperative Bank,3208,8000F4580,8008EEA70,Partnership #2,Arbor Savings Bank,1,8000F5340,800AA5D20,Corporation #1
2,2022-01-09 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,...,National Bank of Fort Wayne,3209,8000F4670,800FBB3A0,Partnership #3,National Bank of Fort Wayne,3209,8000F4670,800FBB3A0,Partnership #3
3,2022-01-09 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,...,National Bank of the East,12,8000F5030,800C0EF20,Sole Proprietorship #1,National Bank of the East,12,8000F5030,800C0EF20,Sole Proprietorship #1
4,2022-01-09 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,...,National Bank of Laramie,10,8000F5200,800C3EC10,Partnership #4,National Bank of Laramie,10,8000F5200,800C3EC10,Partnership #4


## 4. Análise da Distribuição Temporal

In [15]:
# Análise da distribuição por mês
df['Year_Month'] = df['Timestamp'].dt.to_period('M')
distribuicao_mensal = df.groupby('Year_Month').agg({
    'Timestamp': 'count',
    'Is Laundering': 'sum'
}).rename(columns={'Timestamp': 'Total_Transacoes', 'Is Laundering': 'Total_Lavagem'})

distribuicao_mensal['Percentual_Lavagem'] = (
    distribuicao_mensal['Total_Lavagem'] / distribuicao_mensal['Total_Transacoes'] * 100
).round(2)

print("Distribuição mensal dos dados:")
print(distribuicao_mensal)

Distribuição mensal dos dados:
            Total_Transacoes  Total_Lavagem  Percentual_Lavagem
Year_Month                                                     
2022-01              1114921            322                0.03
2022-02               754449            408                0.05
2022-03               207382            391                0.19
2022-04               207430            407                0.20
2022-05               482650            471                0.10
2022-06               482089            531                0.11
2022-07               482751            497                0.10
2022-08               482773            539                0.11
2022-09               654467            514                0.08
2022-10               208325            442                0.21
2022-11                  396            232               58.59
2022-12                  281            170               60.50


## 5. Divisão Temporal: Treino e Out-of-Time

Vamos dividir os dados utilizando uma proporção de **80% para treino** e **20% para out-of-time**. 
A divisão será feita com base na ordem temporal das transações.

### ⚠️ GAP de Segurança (Anti-Leakage)

Para evitar Data Leakage causado por atraso na marcação de fraude (chargeback delay), 
implementamos um **gap de 7 dias** entre o fim do treino e o início do OOT.

**Exemplo de problema sem gap:**
- Transação fraudulenta em 31/12/2023 às 23:59h
- Chargeback reportado apenas em 05/01/2024
- Se OOT começar em 01/01/2024, o modelo "vê o futuro" (label delay leakage)

**Solução:** Gap temporal de 7 dias entre treino e OOT.

In [16]:
# Ordenar os dados por timestamp
df_sorted = df.sort_values('Timestamp').reset_index(drop=True)

# Definir o ponto de corte temporal (80% para treino, 20% para OOT)
split_ratio = 0.8
split_index = int(len(df_sorted) * split_ratio)

# GAP de segurança para evitar label leakage (chargeback delay)
GAP_DAYS = 7  # 7 dias de segurança

# Obter a data de corte inicial
data_corte_inicial = df_sorted.iloc[split_index]['Timestamp']

# Aplicar o gap: OOT começa GAP_DAYS após a data de corte
data_corte_com_gap = data_corte_inicial + pd.Timedelta(days=GAP_DAYS)

# Realizar a divisão COM GAP
df_treino = df_sorted[df_sorted['Timestamp'] < data_corte_inicial].copy()
df_gap = df_sorted[
    (df_sorted['Timestamp'] >= data_corte_inicial) & 
    (df_sorted['Timestamp'] < data_corte_com_gap)
].copy()
df_oot = df_sorted[df_sorted['Timestamp'] >= data_corte_com_gap].copy()

print("="*80)
print("DIVISÃO TEMPORAL REALIZADA COM GAP DE SEGURANÇA")
print("="*80)

print(f"\n📊 Dataset de Treino (df_treino):")
print(f"   - Total de registros: {len(df_treino):,}")
print(f"   - Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   - Casos de lavagem: {df_treino['Is Laundering'].sum():,}")
print(f"   - Taxa de lavagem: {(df_treino['Is Laundering'].sum() / len(df_treino) * 100):.2f}%")

print(f"\n⚠️  GAP de Segurança (descartado):")
print(f"   - Total de registros: {len(df_gap):,}")
print(f"   - Período: {df_gap['Timestamp'].min()} até {df_gap['Timestamp'].max()}")
print(f"   - Justificativa: Evita label leakage por chargeback delay")

print(f"\n📊 Dataset Out-of-Time (df_oot):")
print(f"   - Total de registros: {len(df_oot):,}")
print(f"   - Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   - Casos de lavagem: {df_oot['Is Laundering'].sum():,}")
print(f"   - Taxa de lavagem: {(df_oot['Is Laundering'].sum() / len(df_oot) * 100):.2f}%")

print(f"\n📅 Datas de Corte:")
print(f"   - Fim do Treino: {df_treino['Timestamp'].max()}")
print(f"   - Gap: {GAP_DAYS} dias")
print(f"   - Início do OOT: {df_oot['Timestamp'].min()}")
print(f"\n✅ Gap temporal implementado com sucesso!")

DIVISÃO TEMPORAL REALIZADA COM GAP DE SEGURANÇA

📊 Dataset de Treino (df_treino):
   - Total de registros: 4,062,463
   - Período: 2022-01-09 00:00:00 até 2022-08-09 16:11:00
   - Casos de lavagem: 3,379
   - Taxa de lavagem: 0.08%

⚠️  GAP de Segurança (descartado):
   - Total de registros: 151,982
   - Período: 2022-08-09 16:12:00 até 2022-08-09 23:59:00
   - Justificativa: Evita label leakage por chargeback delay

📊 Dataset Out-of-Time (df_oot):
   - Total de registros: 863,469
   - Período: 2022-09-09 00:00:00 até 2022-12-09 23:51:00
   - Casos de lavagem: 1,358
   - Taxa de lavagem: 0.16%

📅 Datas de Corte:
   - Fim do Treino: 2022-08-09 16:11:00
   - Gap: 7 dias
   - Início do OOT: 2022-09-09 00:00:00

✅ Gap temporal implementado com sucesso!


## 6. Salvamento dos Datasets

In [17]:
# Definir os caminhos de saída
output_dir = r'C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed'
os.makedirs(output_dir, exist_ok=True)

# Caminhos dos arquivos de saída
treino_path = os.path.join(output_dir, 'df_treino.csv')
oot_path = os.path.join(output_dir, 'df_oot.csv')

# Remover a coluna Year_Month antes de salvar (coluna auxiliar)
df_treino_save = df_treino.drop(columns=['Year_Month'], errors='ignore')
df_oot_save = df_oot.drop(columns=['Year_Month'], errors='ignore')

# Salvar os datasets
print("Salvando datasets...")
df_treino_save.to_csv(treino_path, index=False)
df_oot_save.to_csv(oot_path, index=False)

print(f"\n✅ Datasets salvos com sucesso!")
print(f"\n📁 Arquivos salvos em:")
print(f"   - Treino: {treino_path}")
print(f"   - OOT: {oot_path}")

# Verificar tamanhos dos arquivos
treino_size = os.path.getsize(treino_path) / (1024 * 1024)  # MB
oot_size = os.path.getsize(oot_path) / (1024 * 1024)  # MB

print(f"\n📏 Tamanho dos arquivos:")
print(f"   - Treino: {treino_size:.2f} MB")
print(f"   - OOT: {oot_size:.2f} MB")

Salvando datasets...

✅ Datasets salvos com sucesso!

📁 Arquivos salvos em:
   - Treino: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\df_treino.csv
   - OOT: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\df_oot.csv

📏 Tamanho dos arquivos:
   - Treino: 872.33 MB
   - OOT: 184.83 MB


## 7. Verificação Final

Vamos verificar a consistência dos dados salvos.

In [18]:
# Recarregar os arquivos para verificação
df_treino_verificacao = pd.read_csv(treino_path)
df_oot_verificacao = pd.read_csv(oot_path)

print("="*60)
print("VERIFICAÇÃO DOS ARQUIVOS SALVOS")
print("="*60)

print(f"\n✓ Dataset de Treino carregado:")
print(f"  - Registros: {len(df_treino_verificacao):,}")
print(f"  - Colunas: {len(df_treino_verificacao.columns)}")
print(f"  - Formato: {df_treino_verificacao.shape}")

print(f"\n✓ Dataset Out-of-Time carregado:")
print(f"  - Registros: {len(df_oot_verificacao):,}")
print(f"  - Colunas: {len(df_oot_verificacao.columns)}")
print(f"  - Formato: {df_oot_verificacao.shape}")

print(f"\n✓ Total de registros: {len(df_treino_verificacao) + len(df_oot_verificacao):,}")
print(f"✓ Registros originais: {len(df):,}")
print(f"✓ Diferença: {len(df) - (len(df_treino_verificacao) + len(df_oot_verificacao))}")

print("\n" + "="*60)
print("✅ DIVISÃO TEMPORAL CONCLUÍDA COM SUCESSO!")
print("="*60)

VERIFICAÇÃO DOS ARQUIVOS SALVOS

✓ Dataset de Treino carregado:
  - Registros: 4,062,463
  - Colunas: 21
  - Formato: (4062463, 21)

✓ Dataset Out-of-Time carregado:
  - Registros: 863,469
  - Colunas: 21
  - Formato: (863469, 21)

✓ Total de registros: 4,925,932
✓ Registros originais: 5,078,345
✓ Diferença: 152413

✅ DIVISÃO TEMPORAL CONCLUÍDA COM SUCESSO!
